In [1]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer

MODEL_PATH = "AungMoonLove/bert-log-anomaly-detection"

model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)

model.eval()  # สำคัญมาก


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/732 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [2]:
#1 ทำ preprocessing สำหรับ log (For Model Stage 2)

def add_prefix_token(text): # log data ต้องผ่าน code นี้ก่อน training / inference
    # clean log
    text = text.replace("\t", " ")
    text = text.strip()
    # add token
    if text[0].isalpha() or text[3].isalpha():
        return "[SQL]\n" + text
    else:
        return "[LOG]\n" + text

In [3]:
def predict_log(log_text):
    log_text = add_prefix_token(log_text)
    inputs = tokenizer(
        log_text,
        return_tensors="pt",
        truncation=True,
        padding=True, # ใส่เผื่อเอาไว้ตอน inference มากกว่า 1 log (Batch Size > 1)
        max_length=128
    )

    with torch.no_grad():
        logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=1).item()
        prob = torch.softmax(logits, dim=-1).tolist()[0]

    return "NORMAL" if pred == 1 else "ANOMALY" ,prob

----ตรวจคำตอบ------

In [5]:
# ใช้ log บรรทัดที่ 3 (นับแถวของ column) ด้วย

predict,confidence =predict_log(
    """
SELECT BENCHMARK(100000000, MD5('test'))
    """
)
print(predict,confidence)

# คำตอบที่ถูกคือ normal

ANOMALY [0.9839882254600525, 0.016011729836463928]


In [6]:
# ใช้ log บรรทัดที่ 4 (นับแถวของ column) ด้วย

print(predict_log(
    """
024-01-15 22:15:50,999.888,198.51.100.33,schema_attacker,SELECT SCHEMA_NAME, DEFAULT_CHARACTER_SET_NAME FROM INFORMATION_SCHEMA.SCHEMATA;

    """
))
# คำตอบที่ถูกคือ anomally

('ANOMALY', [0.983609676361084, 0.016390273347496986])
